# 👕 vwear — CatVTON on Kaggle
**الإعدادات المطلوبة قبل التشغيل:**
- Settings → Accelerator → **GPU T4 x2** أو **GPU P100**
- Settings → Internet → **On**
- في Kaggle Secrets أضف:
  - `NGROK_TOKEN` → token بتاعك من ngrok.com
  - `HF_TOKEN` → token بتاعك من huggingface.co (اختياري)


In [1]:
# ════════════════════════════════════════════════
# CELL 1 — Install Dependencies
# ════════════════════════════════════════════════
import subprocess, sys

packages = [
    'fastapi', 'uvicorn[standard]', 'python-multipart',
    'pyngrok', 'nest_asyncio', 'aiofiles',
    'diffusers', 'transformers', 'accelerate',
    'omegaconf', 'einops',
    'xformers',
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.1 MB/s eta 0:00:00
Packages installed!


In [2]:
# ════════════════════════════════════════════════════════════
# CELL 1-B — تثبيت detectron2 + fvcore (شغّلها قبل CELL 4)
# ════════════════════════════════════════════════════════════
import subprocess, sys, torch

# 1. fvcore (dependency بتاعة detectron2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fvcore'], check=True)

# 2. detectron2 — لازم يتبني من السورس عشان يتطابق مع الـ torch version
TORCH_VER  = torch.__version__.split("+")[0]      # e.g. "2.0.0"
CUDA_VER   = torch.version.cuda.replace(".", "")   # e.g. "118"

print(f"torch={TORCH_VER} | cuda={CUDA_VER}")
print("⏳ Installing detectron2 (1-2 min)...")

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    f'detectron2 @ git+https://github.com/facebookresearch/detectron2.git'
], check=True)

print("  detectron2 + fvcore installed!")

# 3. تأكيد
try:
    import fvcore, detectron2
    print(f"  fvcore:     {fvcore.__version__}")
    print(f"  detectron2: {detectron2.__version__}")
except ImportError as e:
    print(f"❌ {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
torch=2.10.0 | cuda=128
⏳ Installing detectron2 (1-2 min)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 8.0 MB/s eta 0:00:00
  detectron2 + fvcore installed!
  fvcore:     0.1.5.post20221221
  detectron2: 0.6


In [3]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 57.5 MB/s eta 0:00:00:00:0100:01


In [4]:
# شغّل السطر ده في cell جديدة
import fastapi, uvicorn, pyngrok, diffusers, transformers
print("  كل حاجة تمام!")

  كل حاجة تمام!


In [5]:
# ════════════════════════════════════════════════
# CELL 2 — Clone CatVTON + Download Model
# ════════════════════════════════════════════════
import os, subprocess, sys

# ── Clone CatVTON ──────────────────────────────
if not os.path.exists('/kaggle/working/CatVTON'):
    subprocess.run(['git', 'clone', 'https://github.com/Zheng-Chong/CatVTON.git',
                    '/kaggle/working/CatVTON'], check=True)
    print('  CatVTON cloned!')
else:
    print('  CatVTON already exists')

# ── Download Model Weights ──────────────────────
MODEL_PATH = '/kaggle/working/catvton_model'

if not os.path.exists(MODEL_PATH):
    print('⏳ Downloading CatVTON weights (~5 min)...')
    from huggingface_hub import snapshot_download

    # HF token من Kaggle Secrets (اختياري بس بيسرّع)
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except:
        hf_token = None

    snapshot_download(
        repo_id='zhengchong/CatVTON',
        local_dir=MODEL_PATH,
        token=hf_token,
        ignore_patterns=['*.md', '*.txt', 'logs/*'],
    )
    print('  Model downloaded!')
else:
    print('  Model already exists')

# ── Print folder contents ───────────────────────
print('\n📁 Model contents:')
for item in sorted(os.listdir(MODEL_PATH)):
    path = os.path.join(MODEL_PATH, item)
    n = len(os.listdir(path)) if os.path.isdir(path) else ''
    print(f'   {"📂" if os.path.isdir(path) else "📄"} {item} {f"({n} files)" if n else ""}')

Cloning into '/kaggle/working/CatVTON'...


  CatVTON cloned!
⏳ Downloading CatVTON weights (~5 min)...


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

  Model downloaded!

📁 Model contents:
   📂 .cache (1 files)
   📄 .gitattributes 
   📂 DensePose (3 files)
   📂 SCHP (2 files)
   📄 config.json 
   📂 dresscode-16k-512 (1 files)
   📂 flux-lora (1 files)
   📂 mix-48k-1024 (1 files)
   📂 vitonhd-16k-512 (1 files)


In [23]:
# ════════════════════════════════════════════════
# CELL 3 — Write Server File
# ════════════════════════════════════════════════

server_code = '''
import sys
sys.path.insert(0, "/kaggle/working/CatVTON")

import os, io, traceback, base64, torch, numpy as np
from PIL import Image
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware

from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker
from diffusers.image_processor import VaeImageProcessor
from utils import resize_and_crop

MODEL_PATH    = "/kaggle/working/catvton_model"
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
WIDTH, HEIGHT = 768, 1024

print(f"🚀 Loading on {DEVICE}...")

pipeline = CatVTONPipeline(
    base_ckpt="runwayml/stable-diffusion-inpainting",
    attn_ckpt=MODEL_PATH,
    attn_ckpt_version="mix",
    weight_dtype=torch.float16,
    device=DEVICE,
    skip_safety_check=True,
)

automasker = AutoMasker(
    densepose_ckpt=os.path.join(MODEL_PATH, "DensePose"),
    schp_ckpt=os.path.join(MODEL_PATH, "SCHP"),
    device=DEVICE,
)

mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True,
)


try:
    pipeline.enable_xformers_memory_efficient_attention()
    print("  xformers enabled!")
except Exception as e:
    print(f"⚠️ xformers not available: {e}")
print(f"  Ready on {DEVICE}")

app = FastAPI(title="vwear CatVTON")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

def read_image(b):
    return Image.open(io.BytesIO(b)).convert("RGB")

def to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode()

def repaint(person, mask, result):
    """يرجّع الوجه والإيدين والبنطلون من الصورة الأصلية"""
    mask_np  = np.array(mask.convert("L")).astype(np.float32) / 255.0
    mask_3ch = np.stack([mask_np] * 3, axis=-1)
    blended  = (np.array(result) * mask_3ch + np.array(person) * (1 - mask_3ch)).astype(np.uint8)
    return Image.fromarray(blended)

@app.get("/health")
async def health():
    return {"status": "ok", "device": DEVICE, "platform": "kaggle"}

@app.post("/tryon")
async def tryon(
    person_image:        UploadFile = File(...),
    cloth_image:         UploadFile = File(...),
    cloth_type:          str        = Form("upper"),
    num_inference_steps: int        = Form(25),
    guidance_scale:      float      = Form(2.5),
    seed:                int        = Form(42),
):
    try:
        person_pil = resize_and_crop(read_image(await person_image.read()), (WIDTH, HEIGHT))
        cloth_pil  = resize_and_crop(read_image(await cloth_image.read()),  (WIDTH, HEIGHT))

        print(f"person:{person_pil.size} | cloth:{cloth_pil.size} | type:{cloth_type}")

        mask_raw     = automasker(person_pil, cloth_type)["mask"]
        mask_blurred = mask_processor.blur(mask_raw, blur_factor=9)

        generator    = torch.Generator(device=DEVICE).manual_seed(seed)
        result_image = pipeline(
            image=person_pil,
            condition_image=cloth_pil,
            mask=mask_blurred,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator,
        )[0]

        result_image = repaint(person_pil, mask_raw, result_image)

        print("  Done!")
        return JSONResponse({"success": True, "result_image": f"data:image/jpeg;base64,{to_b64(result_image)}"})
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))
'''

with open('/kaggle/working/vwear_server.py', 'w') as f:
    f.write(server_code)

print('  Server file written!')

  Server file written!


In [24]:
# ════════════════════════════════════════════════
# CELL 4 — Start Server + ngrok Tunnel
# ════════════════════════════════════════════════
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("NGROK_TOKEN")

import nest_asyncio, threading, uvicorn, sys
from pyngrok import ngrok
nest_asyncio.apply()
sys.path.insert(0, '/kaggle/working/CatVTON')


# ── ngrok token من Kaggle Secrets ──────────────
try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret('NGROK_TOKEN')
    print('✅ NGROK_TOKEN loaded from Kaggle Secrets')
except Exception:
    # لو مش موجود في Secrets، حطه هنا يدوياً
    NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'
    print('⚠️  Using hardcoded token — add it to Kaggle Secrets for safety')

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)

tunnel     = ngrok.connect(7860, 'http')
PUBLIC_URL = tunnel.public_url

print('=' * 55)
print(f'🌐 URL    : {PUBLIC_URL}')
print(f'📡 Health : {PUBLIC_URL}/health')
print(f'👕 Tryon  : {PUBLIC_URL}/tryon')
print('=' * 55)
print(f'\n📋 في .env بتاع الباك:\n   MODEL_API_URL={PUBLIC_URL}')

# ── تشغيل السيرفر في thread منفصل ──────────────
# exec بيلود الـ app object في الـ namespace الحالي
exec(open('/kaggle/working/vwear_server.py').read(), globals())

config = uvicorn.Config(app, host='0.0.0.0', port=7860, log_level='warning')
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print('\n⏳ Server starting... wait 30 seconds then run CELL 5')

✅ NGROK_TOKEN loaded from Kaggle Secrets
🌐 URL    : https://steadfast-uncheck-appraiser.ngrok-free.dev
📡 Health : https://steadfast-uncheck-appraiser.ngrok-free.dev/health
👕 Tryon  : https://steadfast-uncheck-appraiser.ngrok-free.dev/tryon

📋 في .env بتاع الباك:
   MODEL_API_URL=https://steadfast-uncheck-appraiser.ngrok-free.dev
🚀 Loading on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


⚠️ xformers not available: 'CatVTONPipeline' object has no attribute 'enable_xformers_memory_efficient_attention'
  Ready on cuda

⏳ Server starting... wait 30 seconds then run CELL 5


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 7860): address already in use


In [26]:
# ════════════════════════════════════════════════
# CELL 5 — Health Check
# ════════════════════════════════════════════════
import requests, time

try:
    r = requests.get('http://localhost:7860/health', timeout=10)
    print('  Server is UP:', r.json())
    print(f'\n🎉 جاهز! حدّث MODEL_API_URL في .env بـ:\n   {PUBLIC_URL}')
except Exception as e:
    print(f'❌ Not ready yet: {e}')
    print('   استنى 15 ثانية وشغّل الـ cell دي تاني')

  Server is UP: {'status': 'ok', 'device': 'cuda', 'platform': 'kaggle'}

🎉 جاهز! حدّث MODEL_API_URL في .env بـ:
   https://steadfast-uncheck-appraiser.ngrok-free.dev
person:(768, 1024) | cloth:(768, 1024) | type:upper


100%|██████████| 25/25 [00:51<00:00,  2.07s/it]


  Done!


---
## ⚙️ إعداد Kaggle Secrets (مرة واحدة بس)

1. في الـ notebook: **Add-ons → Secrets**
2. أضف:
   - Key: `NGROK_TOKEN` → Value: token بتاعك من [ngrok.com/dashboard](https://dashboard.ngrok.com)
   - Key: `HF_TOKEN` → Value: token بتاعك من [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) *(اختياري)*
3. فعّل الـ toggle جنب كل secret عشان الـ notebook يقدر يقراه

## 📋 ملاحظات مهمة
- الـ session بتاخد **~10 دقايق** للـ setup أول مرة (تحميل الموديل)
- بعد الـ setup، كل try-on بياخد **~30-40 ثانية**
- لو الـ kernel اتوقف، شغّل **CELL 4 بس** (الموديل بيتحمّل مرة واحدة)
- الـ ngrok URL **بيتغير كل session** — حدّث `.env` في الباك
